# Feature Selection: Choosing the Right Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/03-feature-engineering/01_feature_selection.ipynb)

## Objectives
- Understand why feature selection matters
- Learn multiple feature selection techniques
- Implement univariate, tree-based, and correlation methods
- Compare different selection strategies
- Build practical decision framework

## 1. Why Feature Selection?

**Benefits:**
- ✅ Reduces overfitting (fewer features = simpler model)
- ✅ Improves model interpretability
- ✅ Faster training and inference
- ✅ Reduces storage requirements
- ✅ Can improve generalization

**Costs of too many features:**
- ❌ The Curse of Dimensionality
- ❌ Increased noise and irrelevant correlations
- ❌ Computational overhead
- ❌ Harder to interpret model decisions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    SelectKBest, f_classif, chi2, RFE, SelectFromModel
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.stats import spearmanr, pearsonr

np.random.seed(42)
sns.set_theme()

print("✅ Libraries loaded successfully")

## 2. Create Sample Dataset

Customer marketing dataset with mixed quality features

In [ ]:
# Create synthetic dataset with 20 features, only 5 are truly important
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=5,      # Only 5 features are truly predictive
    n_redundant=5,        # 5 features are combinations of informative features
    n_repeated=5,         # 5 features are random
    n_classes=2,
    random_state=42
)

# Create DataFrame with feature names
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nFirst few rows:")
print(df.head())

## 3. Method 1: Univariate Statistical Tests

Evaluate each feature individually using statistical tests

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('target', axis=1), 
    df['target'], 
    test_size=0.2, 
    random_state=42
)

# SelectKBest with f_classif
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)

# Get selected feature indices
selected_features = X_train.columns[selector.get_support()].tolist()
feature_scores = selector.scores_

# Create results dataframe
univariate_results = pd.DataFrame({
    'Feature': feature_names,
    'F-Score': feature_scores
}).sort_values('F-Score', ascending=False)

print("📊 Top 10 Features by F-Score (Univariate):")
print(univariate_results.head(10))

# Visualization
plt.figure(figsize=(12, 6))
plt.barh(range(10), univariate_results['F-Score'].head(10).values)
plt.yticks(range(10), univariate_results['Feature'].head(10).values)
plt.xlabel('F-Score')
plt.title('Top 10 Features by F-Score (Statistical Test)')
plt.tight_layout()
plt.show()

## 4. Method 2: Tree-Based Feature Importance

Use ensemble models to identify important features

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Get feature importances
tree_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("📊 Top 10 Features by Tree Importance (Random Forest):")
print(tree_importance.head(10))

# Visualization
plt.figure(figsize=(12, 6))
plt.barh(range(10), tree_importance['Importance'].head(10).values, color='forestgreen')
plt.yticks(range(10), tree_importance['Feature'].head(10).values)
plt.xlabel('Importance')
plt.title('Top 10 Features by Tree Importance (Random Forest)')
plt.tight_layout()
plt.show()

## 5. Method 3: Recursive Feature Elimination (RFE)

Iteratively remove features and retrain model

In [ ]:
# Prepare data for RFE
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# RFE with Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(lr, n_features_to_select=10, step=1)
rfe.fit(X_train_scaled, y_train)

# Get selected features
rfe_selected = X_train.columns[rfe.support_].tolist()
rfe_results = pd.DataFrame({
    'Feature': feature_names,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print("📊 RFE Results (Top 10):")
print(rfe_results[rfe_results['Selected']].head(10))
print(f"\nSelected features: {rfe_selected}")

## 6. Method 4: L1 Regularization (Lasso)

Feature selection through coefficients

In [ ]:
# SelectFromModel with L1-based estimator
from sklearn.linear_model import LogisticRegression

lr_l1 = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=1000)
selector_l1 = SelectFromModel(lr_l1, prefit=False)
selector_l1.fit(X_train_scaled, y_train)

l1_selected = X_train.columns[selector_l1.get_support()].tolist()

# Get coefficients
lr_l1.fit(X_train_scaled, y_train)
coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': np.abs(lr_l1.coef_[0])
}).sort_values('Coefficient', ascending=False)

print("📊 L1 Regularization - Top 10 by Absolute Coefficient:")
print(coefficients.head(10))
print(f"\nSelected features (non-zero coefficients): {l1_selected}")

## 7. Correlation-Based Selection

Remove highly correlated redundant features

In [ ]:
# Calculate correlation with target
correlations = df.corr()['target'].drop('target').abs().sort_values(ascending=False)

print("📊 Top 10 Features by Correlation with Target:")
print(correlations.head(10))

# Visualize
plt.figure(figsize=(12, 6))
plt.barh(range(10), correlations.head(10).values, color='coral')
plt.yticks(range(10), correlations.head(10).index)
plt.xlabel('Absolute Correlation')
plt.title('Top 10 Features by Correlation with Target')
plt.tight_layout()
plt.show()

## 8. Feature Redundancy Detection

Find and remove highly correlated feature pairs

# Calculate correlation matrix
corr_matrix = X_train.corr().abs()

# Find highly correlated pairs (threshold = 0.8)
threshold = 0.8
correlated_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > threshold:
            correlated_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if correlated_pairs:
    print(f"📊 Highly Correlated Feature Pairs (r > {threshold}):")
    for feat1, feat2, corr_val in sorted(correlated_pairs, key=lambda x: x[2], reverse=True):
        print(f"  {feat1} <-> {feat2}: {corr_val:.3f}")
else:
    print(f"No feature pairs with correlation > {threshold}")

## 9. Comparison: Selection Methods

Compare different methods side-by-side

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Method': ['Univariate (Top 10)', 'Tree Importance (Top 10)', 'RFE (Top 10)', 
               'L1 (Selected)', 'Correlation (Top 10)'],
    'Features': [
        univariate_results['Feature'].head(10).tolist(),
        tree_importance['Feature'].head(10).tolist(),
        rfe_selected,
        l1_selected,
        correlations.head(10).index.tolist()
    ]
})

print("📊 Comparison of Selection Methods:")
for idx, row in comparison.iterrows():
    print(f"\n{row['Method']}:")
    print(f"  {row['Features'][:3]}..." if len(row['Features']) > 3 else f"  {row['Features']}")

## 10. Model Performance: Full vs Selected Features

Impact of feature selection on model accuracy

In [ ]:
# Train models with different feature sets
results = {}

# Full features
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_train, y_train)
results['All Features (20)'] = accuracy_score(y_test, rf_full.predict(X_test))

# Tree-based selected
tree_features = tree_importance['Feature'].head(10).tolist()
rf_tree = RandomForestClassifier(n_estimators=100, random_state=42)
rf_tree.fit(X_train[tree_features], y_train)
results['Tree Selection (10)'] = accuracy_score(y_test, rf_tree.predict(X_test[tree_features]))

# Univariate selected
univariate_features = univariate_results['Feature'].head(10).tolist()
rf_uni = RandomForestClassifier(n_estimators=100, random_state=42)
rf_uni.fit(X_train[univariate_features], y_train)
results['Univariate Selection (10)'] = accuracy_score(y_test, rf_uni.predict(X_test[univariate_features]))

# Correlation selected
corr_features = correlations.head(10).index.tolist()
rf_corr = RandomForestClassifier(n_estimators=100, random_state=42)
rf_corr.fit(X_train[corr_features], y_train)
results['Correlation Selection (10)'] = accuracy_score(y_test, rf_corr.predict(X_test[corr_features]))

results_df = pd.DataFrame(list(results.items()), columns=['Method', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False)

print("📊 Model Performance Comparison:")
print(results_df.to_string(index=False))

# Visualization
plt.figure(figsize=(10, 6))
plt.bar(range(len(results_df)), results_df['Accuracy'], color='skyblue')
plt.xticks(range(len(results_df)), results_df['Method'], rotation=45, ha='right')
plt.ylabel('Accuracy')
plt.title('Model Performance: Original vs Selected Features')
plt.ylim([0.8, 1.0])
for i, v in enumerate(results_df['Accuracy']):
    plt.text(i, v + 0.005, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## 11. Decision Framework for Feature Selection

**When to use each method:**

```
START: Feature Selection Decision
  |
  ├─ Question 1: Are features numeric or categorical?
  |  ├─ Mostly numeric → Method: Statistical tests (F-score, correlation)
  |  └─ Mixed → Consider multiple methods
  |
  ├─ Question 2: Do you need interpretability?
  |  ├─ Yes → Use: Tree importance, Univariate tests, Correlation
  |  └─ No → Can use: RFE, L1 regularization
  |
  ├─ Question 3: Do you need linear relationships?
  |  ├─ Yes → Use: L1 (Lasso), Correlation-based
  |  └─ No → Use: Tree-based importance, RFE
  |
  └─ Question 4: Is computational efficiency critical?
     ├─ Yes → Use: Univariate tests, Correlation (fastest)
     └─ No → Can afford: RFE, Tree importance
```

## 12. Common Mistakes & Best Practices

**❌ Mistakes to avoid:**
1. Feature selection BEFORE train-test split (leakage!)
2. Using test set performance for selection
3. Selecting features based on correlation alone (misses interactions)
4. Over-selecting features (curse of dimensionality)
5. Not domain validation of selected features

**✅ Best practices:**
1. Always split data first, then select from training set only
2. Use multiple selection methods and compare
3. Domain experts should validate selections
4. Use recursive elimination for interaction detection
5. Combine statistical and model-based methods
6. Monitor test set performance, not training performance

## 13. Summary

### Key Takeaways:
- **Feature Selection Techniques**: Univariate, tree-based, RFE, L1, correlation
- **Univariate**: Fast, interpretable, misses interactions
- **Tree-based**: Captures interactions, handles non-linearities
- **RFE**: Computationally expensive but thorough
- **L1**: Good for linear models, automatic feature selection
- **Always validate selections** with held-out test set
- **No single best method** - use multiple and compare

### Next Steps:
1. Extract new features (Feature Extraction)
2. Transform existing features (Feature Transformation)
3. Combine with domain knowledge for best results